%md 
# init notebook setting up the backend. 

Do not edit the notebook, it contains import and helpers for the demo

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=7474654018043568&notebook=%2F_resources%2F00-init&demo_name=computer-vision-pcb&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fcomputer-vision-pcb%2F_resources%2F00-init&version=1&user_hash=bbdbed5f785ffc3e7d74bf8bb0e97e3e6421d6ad205d21255f112e10dfba5ca0">

## Demo initialization

In [0]:
dbutils.widgets.dropdown("reset_all_data", "false", ["true", "false"], "Reset all data")
reset_all_data = dbutils.widgets.get("reset_all_data") == "true"

In [0]:
%run ../config

In [0]:
%run ./00-global-setup-v2

In [0]:
DBDemos.setup_schema(catalog, db, reset_all_data, volume_name)
volume_folder =  f"/Volumes/{catalog}/{db}/{volume_name}"

In [0]:
import os
import mlflow
import pandas as pd
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, col, regexp_extract, rand, to_timestamp, initcap, sha1
from pyspark.sql.functions import pandas_udf, PandasUDFType, input_file_name, col

## Data initialization (optional)

In [0]:
if reset_all_data or DBDemos.is_folder_empty(volume_folder+"/labels") or DBDemos.is_folder_empty(volume_folder+"/images"):
  # data generation on another notebook to avoid installing libraries (takes a few seconds to setup pip env)
  print(f"Loading raw data under {volume_folder} , please wait a few minutes as we extract all images...")
  # Run the 01-load-data notebook in the _resources folder to load the data from S3 to the volume
  path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
  parent_count = path[path.rfind("computer-vision-dl"):].count('/') - 1
  prefix = "./" if parent_count == 0 else parent_count*"../"
  prefix = f'{prefix}_resources/'
  dbutils.notebook.run(prefix+"01-load-data", 600, {"volume_folder": volume_folder})
else:
  print("data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.")

## Example code: lightning dataloader from hugging face dataset example

If your dataset is small, you could also load it from your spark dataframe with the huggingface dataset library:

In [0]:
def define_lightning_dataset_moduel():

  import pytorch_lightning as pl
  from datasets import Dataset
  class DeltaDataModuleHF(pl.LightningDataModule):
      from torch.utils.data import random_split, DataLoader
      def __init__(self, df, batch_size: int = 64):
          super().__init__()
          # For big dataset, you can use IterableDataset.from_spark()
          self.dataset = Dataset.from_spark(df.select('content', 'label'))
          self.splits = self.dataset.train_test_split(test_size=0.1)
          self.batch_size = batch_size
          self.transform = tf.Compose([
                  tf.Lambda(lambda b: Image.open(io.BytesIO(b)).convert("RGB")),
                  tf.Resize(256),
                  tf.CenterCrop(224),
                  tf.ToTensor(),
                  tf.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
              ])
          
          self.train_ds = self.splits['train'].map(lambda e: {'content': self.transform(e['content']), 'label': e['label']})
          self.train_ds.set_format(type='torch')
          self.val_ds = self.splits['test'].map(lambda e: {'content': self.transform(e['content']), 'label': e['label']})
          self.val_ds.set_format(type='torch')


      def setup(self, stage: str):
          print(f"preparing dataset, stage: {stage}")

      def train_dataloader(self):
          return torch.utils.data.DataLoader(self.train_ds, batch_size=self.batch_size, num_workers=8)
        
      def val_dataloader(self):
          return torch.utils.data.DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=8)

      def test_dataloader(self):
          return torch.utils.data.DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=8)

In [0]:
#Force torch to local filestore to properly support serverless workspaces
try:
    import os
    import tempfile
    import torch

    file_store_path = os.path.join(tempfile.gettempdir(), os.environ["VIRTUAL_ENV"].split("/")[-1])
    store = torch.distributed.FileStore(file_store_path, world_size=1)
    torch.distributed.init_process_group(backend="gloo", rank=0, world_size=1, store=store)
except:
    pass